# Credit Card Fraud Detection using KaggleHub, SMOTE and XGBoost

This notebook downloads the Kaggle dataset directly using `kagglehub`, then trains XGBoost and explores threshold tuning.


In [ ]:
# Install once if needed:
# %pip install kagglehub pandas numpy matplotlib scikit-learn imbalanced-learn xgboost

import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


## 1. Download and load the Kaggle dataset

`kagglehub.dataset_download()` downloads the dataset and returns its local folder. The code then locates the CSV file automatically.


In [ ]:
dataset_path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
print('Dataset folder:', dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, '**', '*.csv'), recursive=True)
if not csv_files:
    raise FileNotFoundError('No CSV file found.')

csv_path = next((p for p in csv_files if os.path.basename(p).lower() == 'creditcard.csv'), csv_files[0])
df = pd.read_csv(csv_path)

print('Loaded file:', csv_path)
print('Dataset shape:', df.shape)
display(df.head())


## 2. Inspect class imbalance

`Class = 0` means legitimate and `Class = 1` means fraud. Fraud is rare, so accuracy alone can be misleading.


In [ ]:
print(df['Class'].value_counts())
print('\nPercentages:')
print((df['Class'].value_counts(normalize=True) * 100).round(4))

df['Class'].value_counts().plot(kind='bar', title='Class Distribution')
plt.xlabel('Class: 0 = Legitimate, 1 = Fraud')
plt.ylabel('Number of transactions')
plt.show()


## 3. Split features and target

`X` contains transaction features. `y` contains the fraud label. Stratification preserves the approximate fraud ratio in both sets.


In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)


## 4. Apply SMOTE to training data only

SMOTE creates synthetic examples of the minority class. We do not apply it to the test set because the test set must represent the original data distribution.


In [ ]:
print('Before SMOTE:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(pd.Series(y_train_resampled).value_counts())


## 5. Train XGBoost

XGBoost builds decision trees sequentially. Each new tree tries to correct errors made by earlier trees.


In [ ]:
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train_resampled, y_train_resampled)
print('XGBoost training completed.')


## 6. Predict fraud probabilities

`predict_proba()` returns class probabilities. Column `1` is the estimated probability of fraud.


In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
results = pd.DataFrame({'actual_class': y_test.values, 'fraud_probability': y_prob})
display(results.head(10))


## 7. Evaluate ROC-AUC and PR-AUC

ROC-AUC measures ranking quality. PR-AUC is particularly informative when fraud is rare because it focuses on precision and recall.


In [ ]:
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print(f'PR-AUC:  {average_precision_score(y_test, y_prob):.4f}')


## 8. Tune the decision threshold

The default threshold is usually `0.5`. A lower threshold may catch more fraud, but it can also create more false positives.


In [ ]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)
    threshold_results.append({
        'threshold': threshold,
        'precision': precision_score(y_test, y_pred_threshold, zero_division=0),
        'recall': recall_score(y_test, y_pred_threshold, zero_division=0),
        'f1_score': f1_score(y_test, y_pred_threshold, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))


## 9. Evaluate a selected threshold

The threshold `0.30` is only a demonstration. A real threshold should be selected using validation data and the cost of false positives and false negatives.


In [ ]:
chosen_threshold = 0.30
y_pred = (y_prob >= chosen_threshold).astype(int)

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred, zero_division=0))


## 10. Inspect feature importance

Feature importance shows which features the XGBoost model relied on. The `V1` to `V28` columns are anonymized, so their names do not have direct business meanings.


In [ ]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.gca().invert_yaxis()
plt.xlabel('Feature importance')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()


## Main conclusions

- KaggleHub downloads the dataset directly from Kaggle.
- SMOTE balances only the training data.
- XGBoost learns patterns using sequential decision trees.
- Threshold tuning controls the precision-recall trade-off.
- ROC-AUC and PR-AUC evaluate probability-ranking performance.
- Feature importance shows which anonymized features the model uses.
